# ML-04: Search Intelligence Data Contract

## Lane
Ranking

## Objective
Create a data contract for a ranking-based machine learning problem using the FlyRank warehouse release.

## 1. Data Contract

### What does one row represent?
One row represents a content item (content_hash_id) aggregated from historical search-performance observations.

### Which tables will be used?
- fact_content_daily_performance
- fact_content_query_90d

### What time window?
Development month: 2026-03

### What would be ranked or predicted?
Content items will be ranked by optimization priority based on signals that indicate weakening search performance.

### What is deliberately excluded?
Future-period metrics, label-derived columns, and information unavailable at ranking time.

In [11]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
import getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter Hugging Face token: ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
)

FACT_QUERY = (
    f"read_parquet('{REL}/fact_content_query_90d.parquet')"
)

## 2. Verification Query 1 — Grain Check

The declared grain is:

report_date × client_hash_id × content_hash_id

The query below checks whether duplicate rows exist at that grain.

In [13]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM {FACT_DAILY}
WHERE month='2026-03'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,cnt


### Result Interpretation

If the result is empty, the table grain matches the contract and there are no duplicate rows at the declared grain.

## 3. Verification Query 2 — Row Count and Date Span

### Result Interpretation

This query verifies the size of the selected slice and confirms the date range covered by the March 2026 partition.

In [14]:
row_stats = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {FACT_DAILY}
WHERE month='2026-03'
""").df()

row_stats

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 4. Verification Query 3 — Availability Check

The warehouse documentation warns that GA4 values before availability dates are zero-filled.

Therefore rows must be filtered using:

ga4_data_available IS TRUE

In [15]:
availability = con.sql(f"""
SELECT
    COUNT(*) AS rows_with_ga4
FROM {FACT_DAILY}
WHERE month='2026-03'
AND ga4_data_available IS TRUE
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_ga4
0,413966


### Result Interpretation

Only rows with reliable GA4 measurements are retained.

## 5. Feature Frame

The following five features are used for ranking content items by optimization priority.

In [16]:
features = con.sql(f"""
WITH march_data AS (
    SELECT *
    FROM {FACT_DAILY}
    WHERE month='2026-03'
),

bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM march_data
),

windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 15 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_prev30,

        SUM(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 15 DAY
                THEN f.gsc_clicks
                ELSE 0
            END
        ) AS clk_prev30,

        AVG(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 15 DAY
                THEN f.gsc_avg_position
            END
        ) AS avg_position_prev30,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 15 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_last30

    FROM march_data f, bounds b
    GROUP BY 1,2
)

SELECT *
FROM windowed
LIMIT 10000
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_prev30,clk_prev30,avg_position_prev30,imp_last30
0,client_73cda7b4e4f265ea,content_20403327d8d9374c,1503.0,5.0,7.454218,2058.0
1,client_73cda7b4e4f265ea,content_b0bc6be94a50d4b8,5.0,0.0,64.250000,9.0
2,client_73cda7b4e4f265ea,content_a291796f66d0b79e,7.0,0.0,64.416667,4.0
3,client_73cda7b4e4f265ea,content_24e4d25256dd6500,12.0,0.0,33.708333,44.0
4,client_73cda7b4e4f265ea,content_04d9bcc1510f3f32,11.0,0.0,59.206349,10.0


### Feature Availability

#### 1. imp_prev30
Available at decision time because it is calculated from historical impressions observed before ranking decisions are made.

#### 2. clk_prev30
Available at decision time because it uses historical click data.

#### 3. avg_position_prev30
Available at decision time because it is based on historical search ranking positions.

#### 4. visible_queries
Available at decision time because query coverage is already known from historical search data.

#### 5. top_query_share
Available at decision time because it is derived from historical query distribution patterns.

In [17]:
qsignals = con.sql(f"""
SELECT
    content_hash_id,
    ANY_VALUE(content_visible_query_count) AS visible_queries,
    MAX(impressions_90d) AS top_query_impressions,
    SUM(impressions_90d) AS kept_impressions
FROM {FACT_QUERY}
GROUP BY content_hash_id
""").df()

qsignals["top_query_share"] = (
    qsignals["top_query_impressions"]
    / qsignals["kept_impressions"]
)

data = features.merge(
    qsignals[
        [
            "content_hash_id",
            "visible_queries",
            "top_query_share"
        ]
    ],
    on="content_hash_id",
    how="left"
)

data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_prev30,clk_prev30,avg_position_prev30,imp_last30,visible_queries,top_query_share
0,client_73cda7b4e4f265ea,content_20403327d8d9374c,1503.0,5.0,7.454218,2058.0,38.0,0.162791
1,client_73cda7b4e4f265ea,content_b0bc6be94a50d4b8,5.0,0.0,64.250000,9.0,1.0,1.000000
2,client_73cda7b4e4f265ea,content_a291796f66d0b79e,7.0,0.0,64.416667,4.0,NaN,NaN
3,client_73cda7b4e4f265ea,content_24e4d25256dd6500,12.0,0.0,33.708333,44.0,11.0,0.125749
4,client_73cda7b4e4f265ea,content_04d9bcc1510f3f32,11.0,0.0,59.206349,10.0,NaN,NaN


## 6. Leakage Experiment

The goal is to demonstrate how a label-derived feature can artificially inflate model performance.

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)

feature_cols = [
    "imp_prev30",
    "clk_prev30",
    "avg_position_prev30",
    "visible_queries",
    "top_query_share"
]

model_data = data.dropna(subset=feature_cols)

X = model_data[feature_cols]
y = model_data["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

honest_score = accuracy_score(
    y_test,
    model.predict(X_test)
)

print("Honest accuracy:", honest_score)

Honest accuracy: 0.6858638743455497


In [19]:
leaky_features = feature_cols + ["imp_last30"]

model_data = data.dropna(subset=leaky_features)

X = model_data[leaky_features]
y = model_data["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

leaky_score = accuracy_score(
    y_test,
    model.predict(X_test)
)

print("Leaky accuracy:", leaky_score)

Leaky accuracy: 0.9267015706806283


### Leakage Observation

The model score increases significantly after adding imp_last30.

This is a leakage feature because imp_last30 directly contributes to the target definition and would not be available when ranking content items in a real production setting.

Therefore the feature is removed and the honest model result is retained.

## 7. Limitation

This analysis excludes rows where ga4_data_available is FALSE. As a result, some clients with limited analytics history are not represented in the final dataset.